In [ ]:
# Author: Mudit Airan
# Building an e-commerce product listing automation system


import re
import json
import time
import base64
import os
from openai import OpenAI
from openai import OpenAI, APIError, APIConnectionError, RateLimitError, APITimeoutError
from dotenv import load_dotenv


# Import the library that lets us load datasets easily
from datasets import load_dataset

# Load the dataset (small version, so it downloads faster)
dataset = load_dataset("ashraq/fashion-product-images-small", split="train")

# Let's look at just the first 5 products to start small
sample_products = dataset.select(range(5))

# Create a simple, organized structure for each product
products = []
for item in sample_products:
    product = {
        "name": item["productDisplayName"],
        "category": item["masterCategory"],
        "subcategory": item["subCategory"],
        "color": item["baseColour"],
        "image": item["image"]  # this is a PIL image object
    }
    products.append(product)


# Create a folder to store product images
os.makedirs("product_images", exist_ok=True)

# This will hold our final list of product info (JSON-friendly)
product_list = []

for i, p in enumerate(products):
    # Save the image as a file, e.g. product_images/product_0.jpg
    image_path = f"product_images/product_{i}.jpg"
    p["image"].save(image_path)

    # Build a dictionary with just text/number data (JSON-safe)
    product_list.append({
        "id": i,
        "name": p["name"],
        "category": p["category"],
        "subcategory": p["subcategory"],
        "color": p["color"],
        "image_path": image_path
    })

# Save everything into products.json
with open("products.json", "w") as f:
    json.dump(product_list, f, indent=4)

print("Saved products.json with", len(product_list), "products")

# Print out the info (without the image, since it's not text)
for p in products:
    print(p["name"], "|", p["category"], "|", p["subcategory"], "|", p["color"])


# Check you have 5 products
# print(len(products))

# Check you can view an image
# products[0]["image"]


def encode_image(image_path):
    # Open the image file in "read binary" mode
    with open(image_path, "rb") as image_file:
        # Read the raw bytes and convert them to a base64 text string
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string

# Test it on our first product's image
image_path = product_list[0]["image_path"]
base64_image = encode_image(image_path)

print(base64_image[:100])  # print just the first 100 characters (it's a long string!)

# Check the type and length
print(type(base64_image))       # should be <class 'str'>
print(len(base64_image))        # should be a large number (thousands of characters)


def create_prompt(product):
    prompt = f"""
You are an expert e-commerce copywriter. Using the image provided and the product details below, write a professional product listing.

Product Details:
- Name: {product['name']}
- Category: {product['category']}
- Subcategory: {product['subcategory']}
- Color: {product['color']}

Please generate the listing in the following format:

Title: (a catchy, SEO-friendly product title, under 70 characters)

Description: (a detailed 3-4 sentence description highlighting style, use-case, and appeal)

Key Features:
- (feature 1)
- (feature 2)
- (feature 3)

SEO Keywords: (5-7 comma-separated keywords a customer might search for)
"""
    return prompt

# Test it on our first product
#test_prompt = create_prompt(product_list[0])
#print(test_prompt)


# Load variables from the .env file into the environment
load_dotenv()

# Replace this with the actual full path to your .env file
# load_dotenv(dotenv_path="/Users/muditairan/Desktop/Ironhack/Lab - Day 4/API-Calling-to-ChatGPT/.env")

# Read the key from the environment (not typed directly in code)
api_key = os.getenv("OPENAI_API_KEY")

# Create the client using that key
client = OpenAI(api_key=api_key)

# Check the key loaded correctly (don't print the actual key for safety!)
#print("Key loaded:", api_key is not None)
#print("Key starts with:", api_key[:5] if api_key else "NOT FOUND")



# ---- Step 5: Call the API ----
def get_product_listing(product):
    base64_image = encode_image(product["image_path"])
    prompt = create_prompt(product)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]
            }
        ],
        max_tokens=500
    )
    return response.choices[0].message.content

# ---- Parse text response into JSON ----
def parse_listing_to_json(result_text):
    listing = {}

    title_match = re.search(r"Title:\s*(.+)", result_text)
    listing["title"] = title_match.group(1).strip() if title_match else ""

    desc_match = re.search(r"Description:\s*(.+?)(?=Key Features:)", result_text, re.DOTALL)
    listing["description"] = desc_match.group(1).strip() if desc_match else ""

    features_match = re.search(r"Key Features:\s*(.+?)(?=SEO Keywords:)", result_text, re.DOTALL)
    if features_match:
        features_block = features_match.group(1)
        listing["key_features"] = [
            line.strip("- ").strip()
            for line in features_block.split("\n")
            if line.strip().startswith("-")
        ]
    else:
        listing["key_features"] = []

    seo_match = re.search(r"SEO Keywords:\s*(.+)", result_text)
    if seo_match:
        listing["seo_keywords"] = [kw.strip() for kw in seo_match.group(1).split(",")]
    else:
        listing["seo_keywords"] = []

    return listing

# ---- Step 6/7: Batch process products ----
def process_all_products(product_list):
    all_listings = []

    for product in product_list:
        print(f"Processing product {product['id']}: {product['name']}...")

        try:
            result = get_product_listing(product)          # call API
            listing = parse_listing_to_json(result)         # parse text -> JSON

            listing["product_id"] = product["id"]
            listing["image_path"] = product["image_path"]

            all_listings.append(listing)
            print(f"✅ Success for product {product['id']}")

        except Exception as e:
            print(f"❌ Failed for product {product['id']}: {e}")
            all_listings.append({
                "product_id": product["id"],
                "image_path": product["image_path"],
                "error": str(e)
            })

        time.sleep(1)

    return all_listings

# ---- Run on 10 products ----
product_list_sample = product_list[:10]
all_listings = process_all_products(product_list_sample)

# ---- Save results ----
with open("generated_listings.json", "w") as f:
    json.dump(all_listings, f, indent=4)

print(f"\nDone! Processed {len(all_listings)} products.")


with open("generated_listings.json", "r") as f:
    data = json.load(f)

# Show first successful listing with its parsed fields
for item in data:
    if "error" not in item:
        print(json.dumps(item, indent=4))
        break





Saved products.json with 5 products
Turtle Check Men Navy Blue Shirt | Apparel | Topwear | Navy Blue
Peter England Men Party Blue Jeans | Apparel | Bottomwear | Blue
Titan Women Silver Watch | Accessories | Watches | Silver
Manchester United Men Solid Black Track Pants | Apparel | Bottomwear | Black
Puma Men Grey T-shirt | Apparel | Topwear | Grey
/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBwcJCQgKDBQNDAsLDBkSEw8UHRofHh0aHBwgJC4nICIsIxwcKDcpLDAx
<class 'str'>
2388
Processing product 0: Turtle Check Men Navy Blue Shirt...
✅ Success for product 0
Processing product 1: Peter England Men Party Blue Jeans...
✅ Success for product 1
Processing product 2: Titan Women Silver Watch...
✅ Success for product 2
Processing product 3: Manchester United Men Solid Black Track Pants...
✅ Success for product 3
Processing product 4: Puma Men Grey T-shirt...
✅ Success for product 4

Done! Processed 5 products.
{
    "title": "** Stylish Navy Blue Turtle Check Men\u2019s Shirt for Effortless Elegance",
  